# Ingest and extract into a schema

**The job.** Records arrive in three different shapes. Get them all into one
schema. Keep the ones that fit. Say exactly why the others did not.

The second half matters more than the first. Anyone can parse the good rows.
The question is what happens to the rest.

**In:** a file of mixed-format records.
**Out:** rows that match the schema, and rows that do not, with reasons.
**Files:** accepted.jsonl, rejected.jsonl, schema.json.

In [1]:
try:
    import browsergraph  # noqa: F401
except ImportError:
    %pip install -q "browsergraph @ git+https://github.com/aidonerightcorp/browsergraph.git"

import json, pathlib
from dataclasses import replace

from browsergraph import execute, viz
from browsergraph.compile import compile_route
from browsergraph.manifest import NodeManifest, PortSpec
from browsergraph.workbench import Edge, NodeCandidate, StageDefinition, WorkbenchDefinition

# A fresh folder each run. Left-over files from a previous run make the "what
# did this produce" list a lie, and that list is half the point here.
import shutil
WORK = pathlib.Path("work")
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir()

def node(node_id, capability, takes, gives, **kw):
    """Describe one node. Ports are (name, type) pairs."""
    return NodeManifest(
        id=node_id, kind="function", description=f"{capability} via {node_id}",
        capabilities=(capability,),
        inputs=tuple(PortSpec(n, t) for n, t in takes),
        outputs=tuple(PortSpec(n, t) for n, t in gives), **kw)

def stage(sid, name, takes, gives, capability, candidates):
    """Describe one step of the job, and what could do it."""
    return StageDefinition(
        id=sid, name=name, required_capabilities=(capability,),
        inputs=tuple(PortSpec(n, t) for n, t in takes),
        outputs=tuple(PortSpec(n, t) for n, t in gives),
        success=f"{name} produced its declared output",
        candidates=tuple(candidates))

def build(title, task, stages, nodes, edges=()):
    """Put it together and check it before anything runs."""
    bench = WorkbenchDefinition(
        title=title, task=task, stages=tuple(stages), nodes=tuple(nodes),
        edges=tuple(edges),
        candidates=tuple(NodeCandidate(id=n.id, node_id=n.id) for n in nodes))
    problems = bench.validate()
    print("problems:", problems if problems else "none")
    return bench

print("ready")

ready


## The input

Nine records. Some JSON, some `key=value`, some CSV. Two are broken in ways
worth catching: a bad date and a missing required field.

In [2]:
RAW = """{"id": "A1", "email": "ana@example.com", "signed_up": "2026-01-14", "plan": "pro"}
{"id": "A2", "email": "bo@example.com", "signed_up": "2026-02-02", "plan": "free"}
id=A3; email=cy@example.com; signed_up=2026-02-11; plan=pro
id=A4; email=not-an-email; signed_up=2026-03-01; plan=free
A5,dee@example.com,2026-03-09,team
A6,eli@example.com,14/03/2026,pro
A7,fay@example.com,2026-03-20,
{"id": "A8", "email": "gus@example.com", "plan": "free"}
{"id": "A9", "email": "hal@example.com", "signed_up": "2026-04-02", "plan": "enterprise"}"""

source = WORK / "records.txt"
source.write_text(RAW)
print(f"{len(RAW.splitlines())} records in {source}")

9 records in work/records.txt


## The schema

Written down first, on purpose. A schema you infer from the data cannot reject
the data.

In [3]:
SCHEMA = {
    "id":        {"type": "string", "required": True,  "pattern": r"^A\d+$"},
    "email":     {"type": "string", "required": True,  "pattern": r"^[^@\s]+@[^@\s]+\.[a-z]+$"},
    "signed_up": {"type": "date",   "required": True,  "format": "YYYY-MM-DD"},
    "plan":      {"type": "enum",   "required": True,  "values": ["free", "pro", "team"]},
}
(WORK / "schema.json").write_text(json.dumps(SCHEMA, indent=2))
for field, rule in SCHEMA.items():
    print(f"  {field:<11} {rule['type']:<8} {'required' if rule['required'] else 'optional'}")

  id          string   required
  email       string   required
  signed_up   date     required
  plan        enum     required


## The steps

Read, work out the shape of each line, parse it, map it onto the schema, check
it, then split into kept and rejected and write both.

In [4]:
nodes = [
    node("read.lines",  "data.read",  [],                    [("out", "Lines")]),
    node("detect.shape","detect",     [("in", "Lines")],     [("out", "Tagged")]),
    node("parse.mixed", "parse",      [("in", "Tagged")],    [("out", "Raw")]),
    node("map.schema",  "map",        [("in", "Raw")],       [("out", "Mapped")]),
    node("check.schema","validate",   [("in", "Mapped")],    [("ok", "Rows"), ("bad", "Rows")]),
    node("write.both",  "write",      [("ok", "Rows"), ("bad", "Rows")], [("out", "Receipt")],
         effects=("file.write",)),
]

stages = [
    stage("read",   "Read the file",   [],                 [("out", "Lines")],  "data.read", ["read.lines"]),
    stage("detect", "Spot the format", [("in", "Lines")],  [("out", "Tagged")], "detect",    ["detect.shape"]),
    stage("parse",  "Parse each line", [("in", "Tagged")], [("out", "Raw")],    "parse",     ["parse.mixed"]),
    stage("map",    "Map to schema",   [("in", "Raw")],    [("out", "Mapped")], "map",       ["map.schema"]),
    stage("check",  "Check each row",  [("in", "Mapped")], [("ok", "Rows"), ("bad", "Rows")], "validate", ["check.schema"]),
    stage("write",  "Write both files",[("ok", "Rows"), ("bad", "Rows")], [("out", "Receipt")], "write", ["write.both"]),
]

edges = [Edge("read", "detect"), Edge("detect", "parse"), Edge("parse", "map"),
         Edge("map", "check"),
         Edge("check", "write", from_port="ok",  to_port="ok"),
         Edge("check", "write", from_port="bad", to_port="bad")]

bench = build("Ingest into a schema",
              "Get mixed-format records into one schema, and say why the rest failed.",
              stages, nodes, edges)

problems: none


The last step takes **two** inputs: the good rows and the bad ones. Both get
written. That is a join, and it is why this is a graph and not a list of steps.

In [5]:
viz.dag(bench)

Figure(svg='<svg viewBox="0 0 1170 226" width="1170" height="226" style="max-width:none" role="img"><defs><marker id="bg79693928-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-end"><path d="M0,0 L10,5 L0,10 z" fill="#8a93a0"/></marker></defs><text x="153.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 0</text><g><rect x="60" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="69" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Read the file</text><text x="69" y="108.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="363.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 1</text><g><rect x="270" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="279" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Spot the format</text><text x="279" y="108.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="573.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 2</text><g><rect x="480" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="489" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Parse each line</text><text x="489" y="108.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="783.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 3</text><g><rect x="690" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="699" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Map to schema</text><text x="699" y="108.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="993.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 4</text><g><rect x="900" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="909" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Check each row</text><text x="909" y="108.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="1203.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 5</text><g><rect x="1110" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="1119" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Write both files</text><text x="1119" y="108.0" font-size="9.5" fill="#68737f">1 candidate</text></g><path d="M246,100.0 C258.0,100.0 258.0,100.0 270,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg79693928-arrow)"/><path d="M456,100.0 C468.0,100.0 468.0,100.0 480,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg79693928-arrow)"/><path d="M666,100.0 C678.0,100.0 678.0,100.0 690,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg79693928-arrow)"/><path d="M876,100.0 C888.0,100.0 888.0,100.0 900,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg79693928-arrow)"/><path d="M1086,100.0 C1098.0,100.0 1098.0,100.0 1110,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg79693928-arrow)"/><text x="1098.0" y="95.0" text-anchor="middle" font-size="9" fill="#68737f">ok</text><path d="M1086,100.0 C1098.0,100.0 1098.0,100.0 1110,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg79693928-arrow)"/><text x="1098.0" y="95.0" text-anchor="middle" font-size="9" fill="#68737f">bad</text></svg>', title='Ingest into a schema — shape', note='a chain. Boxes in the same layer are independent and may run together; every arrow is a typed port-to-port connection.', width=1170, height=226)

In [6]:
import re
from datetime import date

def read_lines():
    return [l for l in source.read_text().splitlines() if l.strip()]

def detect_shape(**kw):
    """Three shapes. Guess from the first character or the separators."""
    out = []
    for line in kw["in"]:
        if line.lstrip().startswith("{"):
            shape = "json"
        elif "=" in line and ";" in line:
            shape = "keyvalue"
        else:
            shape = "csv"
        out.append((shape, line))
    return out

def parse_mixed(**kw):
    rows = []
    for shape, line in kw["in"]:
        if shape == "json":
            rows.append(json.loads(line))
        elif shape == "keyvalue":
            pairs = [p.strip() for p in line.split(";") if p.strip()]
            rows.append(dict(p.split("=", 1) for p in pairs))
        else:
            bits = [b.strip() for b in line.split(",")]
            rows.append(dict(zip(["id", "email", "signed_up", "plan"], bits)))
    return rows

def map_schema(**kw):
    """Keep only the fields the schema knows about, in schema order."""
    return [{field: row.get(field, "") for field in SCHEMA} for row in kw["in"]]

def check_schema(**kw):
    ok, bad = [], []
    for row in kw["in"]:
        reasons = []
        for field, rule in SCHEMA.items():
            value = (row.get(field) or "").strip()
            if rule["required"] and not value:
                reasons.append(f"{field} is missing"); continue
            if rule["type"] == "enum" and value not in rule["values"]:
                reasons.append(f"{field} {value!r} is not one of {rule['values']}")
            if rule["type"] == "date":
                try:
                    date.fromisoformat(value)
                except ValueError:
                    reasons.append(f"{field} {value!r} is not YYYY-MM-DD")
            if "pattern" in rule and value and not re.match(rule["pattern"], value):
                reasons.append(f"{field} {value!r} does not look right")
        (ok if not reasons else bad).append(
            row if not reasons else dict(row, why="; ".join(reasons)))
    return {"ok": ok, "bad": bad}

def write_both(workspace, **kw):
    good = workspace / "accepted.jsonl"
    bad = workspace / "rejected.jsonl"
    good.write_text("\n".join(json.dumps(r) for r in kw["ok"]))
    bad.write_text("\n".join(json.dumps(r) for r in kw["bad"]))
    return {"accepted": len(kw["ok"]), "rejected": len(kw["bad"])}

runtime = execute.Runtime({
    "read.lines": read_lines, "detect.shape": detect_shape,
    "parse.mixed": parse_mixed, "map.schema": map_schema,
    "check.schema": check_schema, "write.both": write_both,
})

plan = compile_route(bench, {s.id: s.candidates[0] for s in bench.leaf_stages})
run = execute.run(plan, runtime, workspace=WORK)
print(run.text())

plan plan:df11d0fdceea1750760d5…
6 steps in 0.004s — ok
  ok   read             0.000s  read.lines
  ok   detect           0.000s  detect.shape
  ok   parse            0.000s  parse.mixed
  ok   map              0.000s  map.schema
  ok   check            0.001s  check.schema
  ok   write            0.001s  write.both  [file.write]
  file /home/username/code_projects/repos/browsergraph/notebooks/work/accepted.jsonl  331 bytes  sha256:fba73ad1b…
  file /home/username/code_projects/repos/browsergraph/notebooks/work/rejected.jsonl  632 bytes  sha256:35f8dd4b4…


## What got in, and what did not

In [7]:
ok = run.values[("check", "ok")]
bad = run.values[("check", "bad")]

print(f"accepted {len(ok)}:")
for row in ok:
    print(f"  {row['id']:<4}{row['email']:<22}{row['signed_up']:<12}{row['plan']}")

print(f"\nrejected {len(bad)}:")
for row in bad:
    print(f"  {row['id'] or '(no id)':<4}{row['email'][:20]:<22}{row['why']}")

accepted 4:
  A1  ana@example.com       2026-01-14  pro
  A2  bo@example.com        2026-02-02  free
  A3  cy@example.com        2026-02-11  pro
  A5  dee@example.com       2026-03-09  team

rejected 5:
  A4  not-an-email          email 'not-an-email' does not look right
  A6  eli@example.com       signed_up '14/03/2026' is not YYYY-MM-DD
  A7  fay@example.com       plan is missing
  A8  gus@example.com       signed_up is missing
  A9  hal@example.com       plan 'enterprise' is not one of ['free', 'pro', 'team']


Three rejected, each for a different reason, each named. That is the useful
output. A run that said "6 of 9 rows loaded" would leave you to find out which
three and why.

In [8]:
print("files written:")
for art in run.artifacts:
    print(f"  {art.path:<34} {art.bytes:>8,} bytes  {art.digest[:18]}…")

files written:
  /home/username/code_projects/repos/browsergraph/notebooks/work/accepted.jsonl      331 bytes  sha256:fba73ad1bfb…
  /home/username/code_projects/repos/browsergraph/notebooks/work/rejected.jsonl      632 bytes  sha256:35f8dd4b4a9…
